In [0]:
import pyspark.sql.functions as func
from pyspark.sql.window import Window

### Read Data Sources

In [0]:
df = spark.read.table(
    'workspace.sales_db_gold.sales'
)

display(df)

Databricks visualization. Run in Databricks to view.

### Aggregations

How Many customers do we have ?

In [0]:
print(
  df.select('customerID')
    .distinct()
    .count()
)

How many male / female customers do we have

In [0]:
display(
    df.groupBy(
        'customerGender'
    ).agg(
        func.countDistinct('customerID').alias('num_customers')
    )
)

How many customers per country do we have ?



In [0]:
display(
    df.groupBy(
        'customerCountry'
    ).agg(
        func.countDistinct('customerID').alias('num_customers')
    ).orderBy(
        func.col('customerCountry').desc()
    )
)

In [0]:
display(
    df.groupBy(
        'customerCountry', 'customerGender'
    ).agg(
        func.countDistinct('customerID').alias('num_customers')
    ).orderBy(
        func.col('customerCountry').desc()
    )
)



### Exploratory Data Analysis

**Exercise**: What has been the city with highest sales ?

**Exercise**: What is the hour day with highest number of sales ?

**Exercise**: What are the most selling products ?

### Windowing

In [0]:
# What is the sales percentage of each product per day


df_dist = df.groupBy(
    'day',
    'product'
).agg(
    func.sum('quantity').alias('total_sales_per_day_product')
)


df_dist = df_dist.withColumn(
    'total_sales_per_day',
    func.sum(
        'total_sales_per_day_product'
    ).over(
        Window.partitionBy(
            'day'
        ).orderBy(
            'day'
        )
    )
)

df_dist = df_dist.withColumn(
    'sales_percentage',
    func.col('total_sales_per_day_product') / func.col('total_sales_per_day')
)


display(
    df_dist.select(
        'day',
        'product',
        'total_sales_per_day_product',
        'total_sales_per_day',
        'sales_percentage'
    ).orderBy(
        func.col('day'),
        func.col('product')
    )
)

Databricks visualization. Run in Databricks to view.

### User Defined Functions UDF

Allows Data Scientists and Data Engineers to define our own functions.

In [0]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

# Define UDF
def rename_size(x):
    if x == 'S':
        return 'small'
    elif x == 'M':
        return 'medium'
    elif x == 'L':
        return 'large'
    else:
        return 'Huge'
    

# Register UDF
rename_size_udf = udf(rename_size, StringType())

# Apply UDF in select
display(
    df.select(
        func.col('franchiseSize').alias('franchiseSize_old'),
        func.col('supplierSize').alias('supplierSize_old'),
        rename_size_udf(func.col('franchiseSize')).alias('franchiseSize'),
        rename_size_udf(func.col('supplierSize')).alias('supplierSize')
    ).orderBy(
        func.col('franchiseSize_old'),
        func.col('supplierSize_old')
    ).where(
        (func.col('franchiseSize_old').isNotNull()) &
        (func.col('supplierSize_old').isNotNull())
    )
)


**Exercise**: What are the ingredients per product ?

### Pivoting

In [0]:

df_rtmp = df.select(
  'dateTime',
  'month',
  'day',
  'product',
  'quantity'
).distinct()

df_rtmp = df_rtmp.withColumn(
  'hour',
  func.hour('dateTime')
)

df_pivot = df_rtmp.groupBy(
  'month',
  'day',
  'hour'
).pivot(
  'product'
).agg(
  func.sum('quantity')
).fillna(0).orderBy(
  'month',
  'day',
  'hour'
)

# Rename Columns
cols = [c for c in df_pivot.columns if c not in ['month', 'day', 'hour']]

for c in cols:
  df_pivot = df_pivot.withColumnRenamed(
    c,
    f'sales_{c.replace(' ', '_')}'
  )

display(df_pivot)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

**Exercise**: Apply logarithm to sales and represent the historgram

### Modeling

Predict next hour sales for a specific product.

In [0]:
PRODUCT = 'Pearly Pies'
p = PRODUCT.replace(' ', '_')

# Compute next hour sales as the target to predict
df_X = df_pivot.withColumn(
    'dummy',
    func.lit('1')
).withColumn(
    f"next_sales_{p}",
    func.lead(
        f"sales_{p}"
    ).over(
        Window.partitionBy(
            'dummy'
        ).orderBy(
            'day',
            'hour'
        )
    )
).drop(
    'dummy'
).dropna()

display(df_X.select(
    'day',
    'hour',
    f'sales_{p}',
    f'next_sales_{p}'
))

In [0]:
import mlflow
import mlflow.spark
import pandas as pd
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
import warnings
from mlflow.models import infer_signature
warnings.filterwarnings("ignore")

(trainDF, testDF) = df_X.randomSplit([0.8, 0.2], seed=42)

TARGET = f"next_sales_{p}"
FEATURES = [c for c in df_X.columns if c != TARGET]

rf = RandomForestRegressor(
  labelCol=f"next_sales_{p}",
  maxBins=20,
  maxDepth=5,
  numTrees=50,
  seed=42
)


pipeline = Pipeline(stages=[rf])
vecAssembler = VectorAssembler(
  inputCols=FEATURES,
  outputCol="features"
)

pipeline = Pipeline(stages=[vecAssembler, rf])

# Train model
with mlflow.start_run(run_name='random-forest') as run:
  # Log params of the model
  mlflow.log_param('maxDepth', rf.getMaxDepth())
  mlflow.log_param('numTrees', rf.getNumTrees())
  
  # Train model
  model = pipeline.fit(trainDF)
  predictions = model.transform(testDF)

  # Log Model
  signature = infer_signature(testDF, predictions)
  mlflow.spark.log_model(
    model, 
    'model', 
    dfs_tmpdir='/Volumes/workspace/default/models',
    registered_model_name='sales-predictions',
    signature=signature)
  
  # Log metrics
  evaluator = RegressionEvaluator(
    labelCol=TARGET,
    predictionCol="prediction"
  )
  rmse = evaluator.setMetricName("rmse").evaluate(predictions)
  r2 = evaluator.setMetricName("r2").evaluate(predictions)
  mlflow.log_metrics(
    {
      "rmse": rmse,
      "r2" : r2
    }
  )
  
  # Log artifacts
  rfModel = model.stages[-1]
  pandasDF = pd.DataFrame(
    list(
      zip(
        FEATURES,
        rfModel.featureImportances,
      )
    ),
    columns=['feature', 'importance']
  ).sort_values(by='importance', ascending=False)

  fig = pandasDF.plot(
    kind='barh',
    x='feature',
    y='importance',
    title=f"Feature importance for {p}"
  ).get_figure()

  mlflow.log_figure(fig, "feature_importance.png")


**Excercise**: Improve the existing model.

  Try new experiments with Cross-Validation
  
  Try another model
  
  Play with hyperparameters....

### Read the model & Make Predictions

In [0]:
import mlflow.spark
import mlflow

mlflow.set_registry_uri("databricks-uc")
spark_model = mlflow.spark.load_model(
    'models:/workspace.default.sales-predictions/1',
    dfs_tmpdir='/Volumes/workspace/default/models')

final_predictions = spark_model.transform(testDF)
display(final_predictions)